# Analyses statistiques descriptives pour le chapitre d'impl?mentation

Ce notebook produit les tableaux et figures utiles pour d?crire le corpus et les briques techniques du syst?me de recommandation. Les analyses sont descriptives : elles caract?risent les donn?es, les r?f?rentiels, pgvector et Neo4j. Elles ne mesurent pas encore la performance du moteur ; les m?triques de performance rel?vent du chapitre d'?valuation.

Les sorties sont export?es vers :

- `outputs/memoire_stats/implementation_deep` pour les tableaux et figures auditables ;
- `rapport/figures/generated/implementation_deep` pour les images ? ins?rer dans le m?moire.

In [1]:
from __future__ import annotations

from pathlib import Path
import json
import re
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from IPython.display import display
except Exception:
    display = print

ROOT = Path.cwd()
if not (ROOT / 'data').exists():
    ROOT = ROOT.parent

PROC = ROOT / 'data' / 'processed'
OUT = ROOT / 'outputs' / 'memoire_stats' / 'implementation_deep'
FIG = ROOT / 'rapport' / 'figures' / 'generated' / 'implementation_deep'
OUT.mkdir(parents=True, exist_ok=True)
FIG.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    'figure.dpi': 130,
    'font.size': 10,
    'axes.titlesize': 12,
    'axes.labelsize': 10,
})

COLORS = {
    'blue': '#2f6fdd',
    'orange': '#f29700',
    'green': '#2d8a57',
    'gray': '#455a64',
    'red': '#b00020',
    'teal': '#008c8c',
}

def save_table(df: pd.DataFrame, name: str) -> Path:
    path = OUT / f'{name}.csv'
    df.to_csv(path, index=False, encoding='utf-8')
    return path


def save_fig(fig, name: str):
    for folder in (OUT, FIG):
        fig.savefig(folder / f'{name}.png', dpi=180, bbox_inches='tight')
    plt.close(fig)


def clean_label(value, fallback='Non renseign?'):
    if value is None:
        return fallback
    text = str(value).strip()
    if text in {'', 'nan', 'None', '<NA>'}:
        return fallback
    return text


def as_list(value):
    if isinstance(value, (list, tuple, set)):
        return list(value)
    if hasattr(value, 'tolist'):
        converted = value.tolist()
        return converted if isinstance(converted, list) else [converted]
    if value is None or str(value) in {'nan', '<NA>', 'None'}:
        return []
    if isinstance(value, str):
        return [x.strip() for x in re.split(r'[,;|/]+', value) if x.strip()]
    return []

print('Racine projet:', ROOT)

Racine projet: D:\DATA SCIENCES\SYSTEME-DE-RECOMMANDATION-HYBRIDE-


## 1. Chargement des donn?es normalis?es

Cette section charge les tables d?j? normalis?es : offres, candidats, groupes m?tiers MEPC et domaines d?taill?s NCF. Le but est de partir des donn?es effectivement utilis?es par le syst?me, pas des donn?es brutes.

In [2]:
offres = pd.read_parquet(PROC / 'offres_normalized.parquet')
candidats = pd.read_parquet(PROC / 'candidats_normalized.parquet')
mepc = pd.read_parquet(PROC / 'mepc_groupes_base.parquet')
ncf = pd.read_parquet(PROC / 'ncf_dom_detailles.parquet')

frames = {
    'Offres normalis?es': offres,
    'Candidats normalis?s': candidats,
    'Groupes de base MEPC': mepc,
    'Domaines d?taill?s NCF': ncf,
}
overview = pd.DataFrame([
    {'bloc': name, 'nombre_lignes': len(df), 'nombre_colonnes': df.shape[1], 'cellules_non_vides_moyenne': round(float(df.notna().mean().mean()), 4)}
    for name, df in frames.items()
])
save_table(overview, '01_vue_ensemble_donnees')
display(overview)

,bloc,nombre_lignes,nombre_colonnes,cellules_non_vides_moyenne
0,Offres normalis?es,7861,30,0.9227
1,Candidats normalis?s,1105,20,0.8595
2,Groupes de base MEPC,209,7,1.0000
3,Domaines d?taill?s NCF,201,6,1.0000


## 2. Qualit? des donn?es utiles au syst?me

L'objectif est de v?rifier la disponibilit? des champs r?ellement mobilis?s par le matching : secteur, ville, niveau NCF, exp?rience, comp?tences et texte ? encoder.

In [3]:
quality_specs = [
    ('offres', offres, ['titre_poste', 'secteur_principal', 'ville_principale', 'ncf_niveau_code', 'experience_min_ans', 'skills_raw', 'text_to_embed']),
    ('candidats', candidats, ['age', 'genre', 'diplome_raw', 'ncf_niveau_final', 'secteur_metier', 'filiere_specialite', 'metier_vise', 'text_to_embed']),
]
quality_rows = []
for source, df, cols in quality_specs:
    for col in cols:
        non_empty = df[col].notna() & (df[col].astype(str).str.strip() != '') & ~df[col].astype(str).isin(['nan', '<NA>', 'None'])
        quality_rows.append({'source': source, 'champ': col, 'taux_renseignement': round(float(non_empty.mean()), 4), 'non_vides': int(non_empty.sum()), 'total': int(len(df))})
quality = pd.DataFrame(quality_rows).sort_values(['source', 'taux_renseignement'], ascending=[True, False])
save_table(quality, '02_qualite_champs_cles')
display(quality)
fig, ax = plt.subplots(figsize=(10, 5.5))
plot_df = quality.copy(); plot_df['label'] = plot_df['source'] + ' - ' + plot_df['champ']; plot_df = plot_df.sort_values('taux_renseignement')
ax.barh(plot_df['label'], plot_df['taux_renseignement'], color=COLORS['blue'])
ax.set_xlim(0, 1); ax.set_xlabel('Taux de renseignement'); ax.set_title('Disponibilit? des champs cl?s pour le matching'); ax.grid(axis='x', alpha=0.25)
save_fig(fig, '02_qualite_champs_cles')

,source,champ,taux_renseignement,non_vides,total
7,candidats,age,1.0000,1105,1105
8,candidats,genre,1.0000,1105,1105
9,candidats,diplome_raw,1.0000,1105,1105
10,candidats,ncf_niveau_final,1.0000,1105,1105
11,candidats,secteur_metier,1.0000,1105,1105
12,candidats,filiere_specialite,1.0000,1105,1105
13,candidats,metier_vise,1.0000,1105,1105
14,candidats,text_to_embed,1.0000,1105,1105
0,offres,titre_poste,1.0000,7861,7861
1,offres,secteur_principal,1.0000,7861,7861


## 3. Structure sectorielle des offres

Cette analyse montre les secteurs qui dominent le corpus d'offres. Elle est importante car les secteurs surrepr?sent?s influencent les r?sultats de recherche et les recommandations.

In [4]:
sector = offres['secteur_principal'].map(clean_label).value_counts().rename_axis('secteur').reset_index(name='n')
sector['part'] = (sector['n'] / len(offres)).round(4)
save_table(sector, '03_offres_par_secteur')
display(sector.head(15))
fig, ax = plt.subplots(figsize=(9.5, 6))
plot_df = sector.head(15).sort_values('n')
ax.barh(plot_df['secteur'], plot_df['n'], color=COLORS['green'])
ax.set_title('Principaux secteurs des offres'); ax.set_xlabel("Nombre d'offres"); ax.grid(axis='x', alpha=0.25)
save_fig(fig, '03_offres_par_secteur')

,secteur,n,part
0,Distribution,546,0.0695
1,Administration,513,0.0653
2,Informatique,430,0.0547
3,Commerce,389,0.0495
4,ONG / Humanitaire,342,0.0435
5,Comptabilité,324,0.0412
6,Santé,307,0.0391
7,Tic,306,0.0389
8,Marketing,296,0.0377
9,Autre,290,0.0369


## 4. R?partition g?ographique des offres

Cette analyse permet d'identifier les villes ou zones o? les opportunit?s sont concentr?es. Elle sert aussi ? discuter la port?e g?ographique du syst?me.

In [5]:
city = offres['ville_principale'].map(clean_label).value_counts().rename_axis('ville').reset_index(name='n')
city['part'] = (city['n'] / len(offres)).round(4)
save_table(city, '04_offres_par_ville')
display(city.head(15))
fig, ax = plt.subplots(figsize=(9.5, 5.5))
plot_df = city.head(15).sort_values('n')
ax.barh(plot_df['ville'], plot_df['n'], color=COLORS['orange'])
ax.set_title('Localisation principale des offres'); ax.set_xlabel("Nombre d'offres"); ax.grid(axis='x', alpha=0.25)
save_fig(fig, '04_offres_par_ville')

,ville,n,part
0,Yaoundé,4049,0.5151
1,Cameroun (Ville Non Précisée),1800,0.2290
2,Douala,671,0.0854
3,Bafoussam,319,0.0406
4,Maroua,276,0.0351
5,Garoua,184,0.0234
6,Bamenda,100,0.0127
7,Kribi,99,0.0126
8,Buéa,91,0.0116
9,Poli,30,0.0038


## 5. Niveaux de formation et exp?rience demand?s

Cette section d?crit les exigences des offres : niveau NCF et exp?rience minimale. Ces variables sont utilis?es ensuite par le graphe et par le score hybride.

In [6]:
ncf_offres = offres['ncf_niveau_code'].dropna().astype(int).value_counts().sort_index().rename_axis('niveau_ncf').reset_index(name='n')
ncf_offres['part'] = (ncf_offres['n'] / len(offres)).round(4)
save_table(ncf_offres, '05_offres_par_niveau_ncf')
exp = offres['experience_min_ans'].dropna().astype(int)
exp_summary = pd.DataFrame([{'n_renseigne': int(exp.shape[0]), 'moyenne': round(float(exp.mean()), 2) if len(exp) else None, 'mediane': round(float(exp.median()), 2) if len(exp) else None, 'min': int(exp.min()) if len(exp) else None, 'max': int(exp.max()) if len(exp) else None}])
save_table(exp_summary, '05_experience_minimale_resume')
display(ncf_offres); display(exp_summary)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
axes[0].bar(ncf_offres['niveau_ncf'].astype(str), ncf_offres['n'], color=COLORS['blue'])
axes[0].set_title('Niveau NCF demand? par les offres'); axes[0].set_xlabel('Niveau NCF'); axes[0].set_ylabel("Nombre d'offres"); axes[0].grid(axis='y', alpha=0.25)
axes[1].hist(exp, bins=max(5, min(12, len(exp.unique()) or 5)), color=COLORS['gray'], edgecolor='white')
axes[1].set_title('Exp?rience minimale demand?e'); axes[1].set_xlabel("Ann?es d'exp?rience"); axes[1].set_ylabel("Nombre d'offres"); axes[1].grid(axis='y', alpha=0.25)
plt.tight_layout(); save_fig(fig, '05_ncf_et_experience_offres')

,niveau_ncf,n,part
0,1,1,0.0001
1,4,393,0.0500
2,5,559,0.0711
3,6,1742,0.2216
4,7,1191,0.1515
5,8,1072,0.1364
6,9,55,0.0070


,n_renseigne,moyenne,mediane,min,max
0,4721,2.2,1.0,0,10


## 6. Structure des candidats

Cette section d?crit les profils candidats : niveau NCF, ?ge et secteurs d?clar?s. Ces variables servent ? comprendre le vivier de profils que le syst?me doit apparier avec les offres.

In [7]:
cand_ncf = candidats['ncf_niveau_final'].dropna().astype(int).value_counts().sort_index().rename_axis('niveau_ncf').reset_index(name='n')
cand_ncf['part'] = (cand_ncf['n'] / len(candidats)).round(4)
save_table(cand_ncf, '06_candidats_par_niveau_ncf')
cand_sector = candidats['secteur_metier'].map(clean_label).value_counts().rename_axis('secteur_metier').reset_index(name='n')
cand_sector['part'] = (cand_sector['n'] / len(candidats)).round(4)
save_table(cand_sector, '06_candidats_par_secteur')
age = pd.to_numeric(candidats['age'], errors='coerce').dropna()
age_summary = pd.DataFrame([{'n_renseigne': int(len(age)), 'moyenne': round(float(age.mean()), 2) if len(age) else None, 'mediane': round(float(age.median()), 2) if len(age) else None, 'min': int(age.min()) if len(age) else None, 'max': int(age.max()) if len(age) else None}])
save_table(age_summary, '06_age_candidats_resume')
display(cand_ncf); display(cand_sector.head(15)); display(age_summary)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].bar(cand_ncf['niveau_ncf'].astype(str), cand_ncf['n'], color=COLORS['teal'])
axes[0].set_title('Niveau NCF des candidats'); axes[0].set_xlabel('Niveau NCF'); axes[0].set_ylabel('Nombre de candidats'); axes[0].grid(axis='y', alpha=0.25)
plot_df = cand_sector.head(12).sort_values('n')
axes[1].barh(plot_df['secteur_metier'], plot_df['n'], color=COLORS['green'])
axes[1].set_title('Principaux secteurs vis?s par les candidats'); axes[1].set_xlabel('Nombre de candidats'); axes[1].grid(axis='x', alpha=0.25)
plt.tight_layout(); save_fig(fig, '06_structure_candidats')

,niveau_ncf,n,part
0,1,71,0.0643
1,3,55,0.0498
2,4,193,0.1747
3,5,359,0.3249
4,6,76,0.0688
5,7,290,0.2624
6,8,60,0.0543
7,9,1,0.0009


,secteur_metier,n,part
0,"Bâtiment, Travaux Publics & Aménagement du Ter...",248,0.2244
1,"Commerce, Vente, Marketing & Distribution",169,0.1529
2,"Finance, Banque, Assurance, Comptabilité & Fis...",123,0.1113
3,"Transport, Logistique & Supply Chain",108,0.0977
4,"Sécurité, Sûreté & Défense",68,0.0615
5,"Industrie, Production & Maintenance",64,0.0579
6,"Conseil, Stratégie & Gestion des Ressources Hu...",56,0.0507
7,"Informatique, Data, Télécommunications & Digital",50,0.0452
8,Parcours Étudiant et Formation,32,0.0290
9,"Énergie, Environnement & Eau",24,0.0217


,n_renseigne,moyenne,mediane,min,max
0,1105,31.25,30.0,16,71


## 7. Comp?tences les plus fr?quentes dans les offres

Les comp?tences fr?quentes donnent une premi?re lecture du march? observ?. Elles servent aussi ? comprendre quels signaux textuels apparaissent souvent dans les embeddings et dans le graphe.

In [8]:
skill_counter = Counter()

for value in offres.get('skills_list', pd.Series(dtype=object)):
    for skill in as_list(value):
        skill = clean_label(skill, fallback='').strip()
        if skill:
            skill_counter[skill] += 1

# Fallback: certaines lectures Parquet peuvent rendre skills_list peu exploitable.
# On recompte alors depuis la colonne texte brute des competences.
if not skill_counter and 'skills_raw' in offres.columns:
    for value in offres['skills_raw'].fillna(''):
        for skill in as_list(value):
            skill = clean_label(skill, fallback='').strip()
            if skill:
                skill_counter[skill] += 1

skills = pd.DataFrame(skill_counter.most_common(30), columns=['competence', 'n'])
if not skills.empty:
    skills['part_offres'] = (skills['n'] / len(offres)).round(4)
else:
    skills = pd.DataFrame(columns=['competence', 'n', 'part_offres'])

save_table(skills, '07_top_competences_offres')
display(skills.head(20))

if not skills.empty:
    fig, ax = plt.subplots(figsize=(9.5, 7))
    plot_df = skills.head(20).sort_values('n')
    ax.barh(plot_df['competence'], plot_df['n'], color=COLORS['orange'])
    ax.set_title('Competences les plus frequentes dans les offres')
    ax.set_xlabel("Nombre d'offres")
    ax.grid(axis='x', alpha=0.25)
    save_fig(fig, '07_top_competences_offres')

,competence,n,part_offres
0,Distribution,546,0.0695
1,FINANCE,546,0.0695
2,Commerce de gros,524,0.0667
3,Commerce de détail,524,0.0667
4,ADMINISTRATION,498,0.0634
5,INFORMATIQUE,431,0.0548
6,COMPTABILITÉ,401,0.0510
7,COMMERCE,400,0.0509
8,ONG,388,0.0494
9,MARKETING,370,0.0471


## 8. Longueur des textes envoy?s au mod?le d'embeddings

Le champ `text_to_embed` est le texte r?ellement envoy? au mod?le d'embeddings. Sa longueur renseigne sur la richesse informationnelle disponible pour la recherche vectorielle.

In [9]:
def text_len_series(df, col='text_to_embed'):
    return df[col].fillna('').astype(str).str.split().map(len)
o_len = text_len_series(offres); c_len = text_len_series(candidats)
len_summary = pd.DataFrame([
    {'source': 'offres', 'n': len(o_len), 'moyenne_mots': round(float(o_len.mean()), 2), 'mediane_mots': round(float(o_len.median()), 2), 'p90_mots': round(float(o_len.quantile(0.9)), 2)},
    {'source': 'candidats', 'n': len(c_len), 'moyenne_mots': round(float(c_len.mean()), 2), 'mediane_mots': round(float(c_len.median()), 2), 'p90_mots': round(float(c_len.quantile(0.9)), 2)},
])
save_table(len_summary, '08_longueur_text_to_embed')
display(len_summary)
fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(o_len, bins=35, alpha=0.65, label='Offres', color=COLORS['blue'])
ax.hist(c_len, bins=25, alpha=0.65, label='Candidats', color=COLORS['orange'])
ax.set_title('Longueur des textes envoy?s au mod?le d?embeddings'); ax.set_xlabel('Nombre de mots'); ax.set_ylabel('Nombre d?observations'); ax.legend(); ax.grid(axis='y', alpha=0.25)
save_fig(fig, '08_longueur_text_to_embed')

,source,n,moyenne_mots,mediane_mots,p90_mots
0,offres,7861,36.04,5.0,127.0
1,candidats,1105,26.30,26.0,31.0


## 9. ?quilibre entre offre et demande par secteurs

Cette analyse rapproche les secteurs des offres et les secteurs vis?s par les candidats. Elle ne mesure pas encore le matching individuel, mais elle montre les ?ventuels d?s?quilibres macro entre opportunit?s et profils.

In [10]:
o_sec = offres['secteur_principal'].map(clean_label).value_counts().rename('offres')
c_sec = candidats['secteur_metier'].map(clean_label).value_counts().rename('candidats')
sector_balance = pd.concat([o_sec, c_sec], axis=1).fillna(0).astype(int).reset_index().rename(columns={'index': 'secteur'})
sector_balance['total'] = sector_balance['offres'] + sector_balance['candidats']
sector_balance['ratio_offres_candidats'] = sector_balance.apply(lambda r: round(r['offres'] / r['candidats'], 3) if r['candidats'] > 0 else None, axis=1)
sector_balance = sector_balance.sort_values('total', ascending=False)
save_table(sector_balance, '09_equilibre_offres_candidats_secteurs')
display(sector_balance.head(20))
fig, ax = plt.subplots(figsize=(10, 6))
plot_df = sector_balance.head(12).sort_values('total'); y = np.arange(len(plot_df))
ax.barh(y - 0.18, plot_df['offres'], height=0.36, label='Offres', color=COLORS['blue'])
ax.barh(y + 0.18, plot_df['candidats'], height=0.36, label='Candidats', color=COLORS['orange'])
ax.set_yticks(y); ax.set_yticklabels(plot_df['secteur']); ax.set_title('Comparaison macro entre secteurs des offres et secteurs vis?s'); ax.set_xlabel('Nombre'); ax.legend(); ax.grid(axis='x', alpha=0.25)
save_fig(fig, '09_equilibre_offres_candidats_secteurs')

,secteur,offres,candidats,total,ratio_offres_candidats
0,Distribution,546,0,546,NaN
1,Administration,513,0,513,NaN
2,Informatique,430,0,430,NaN
3,Commerce,389,0,389,NaN
4,ONG / Humanitaire,342,0,342,NaN
5,Comptabilité,324,0,324,NaN
6,Santé,307,0,307,NaN
7,Tic,306,0,306,NaN
8,Marketing,296,0,296,NaN
9,Autre,290,0,290,NaN


## 10. Composition des r?f?rentiels m?tiers et formations

Cette section d?crit la couverture r?f?rentielle disponible : groupes de base MEPC et domaines d?taill?s NCF. Ces r?f?rentiels apportent la structure m?tier et formation utilis?e par le graphe.

In [11]:
mepc_by_grand = mepc['code_grand_groupe'].map(clean_label).value_counts().rename_axis('code_grand_groupe').reset_index(name='n_groupes_base')
ncf_by_grand = ncf['code_grand_domaine'].map(clean_label).value_counts().rename_axis('code_grand_domaine').reset_index(name='n_domaines_detailles')
save_table(mepc_by_grand, '10_mepc_groupes_base_par_grand_groupe')
save_table(ncf_by_grand, '10_ncf_domaines_detailles_par_grand_domaine')
display(mepc_by_grand.head(20)); display(ncf_by_grand.head(20))
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].bar(mepc_by_grand['code_grand_groupe'].astype(str), mepc_by_grand['n_groupes_base'], color=COLORS['gray'])
axes[0].set_title('Groupes de base MEPC par grand groupe'); axes[0].set_xlabel('Grand groupe MEPC'); axes[0].set_ylabel('Nombre'); axes[0].grid(axis='y', alpha=0.25)
axes[1].bar(ncf_by_grand['code_grand_domaine'].astype(str), ncf_by_grand['n_domaines_detailles'], color=COLORS['teal'])
axes[1].set_title('Domaines d?taill?s NCF par grand domaine'); axes[1].set_xlabel('Grand domaine NCF'); axes[1].set_ylabel('Nombre'); axes[1].grid(axis='y', alpha=0.25)
plt.tight_layout(); save_fig(fig, '10_referentiels_mepc_ncf')

,code_grand_groupe,n_groupes_base
0,7,50
1,3,34
2,6,32
3,4,28
4,2,23
5,1,18
6,8,14
7,5,10


,code_grand_domaine,n_domaines_detailles
0,5,29
1,7,25
2,10,25
3,4,24
4,8,20
5,2,17
6,6,17
7,1,16
8,3,14
9,9,14


## 11. Composition de pgvector et de Neo4j

Cette section reprend les r?sultats des diagnostics de stockage lorsqu'ils sont disponibles. Elle sert ? documenter les briques techniques aliment?es par les donn?es.

In [12]:
existing = ROOT / 'outputs' / 'memoire_stats' / 'chapter3'
pg_counts = pd.read_csv(existing / 'pgvector_counts.csv') if (existing / 'pgvector_counts.csv').exists() else pd.DataFrame()
neo_nodes = pd.read_csv(existing / 'neo4j_node_counts.csv') if (existing / 'neo4j_node_counts.csv').exists() else pd.DataFrame()
neo_totals = pd.read_csv(existing / 'neo4j_totals.csv') if (existing / 'neo4j_totals.csv').exists() else pd.DataFrame()
if not pg_counts.empty:
    save_table(pg_counts, '11_pgvector_counts'); display(pg_counts)
    fig, ax = plt.subplots(figsize=(9, 5)); plot_df = pg_counts.sort_values('n')
    ax.barh(plot_df['entity_kind'], plot_df['n'], color=COLORS['blue']); ax.set_title('Entit?s index?es dans pgvector'); ax.set_xlabel('Nombre d?embeddings'); ax.grid(axis='x', alpha=0.25); save_fig(fig, '11_pgvector_counts')
if not neo_nodes.empty:
    save_table(neo_nodes, '11_neo4j_node_counts'); display(neo_nodes.head(20))
    fig, ax = plt.subplots(figsize=(9, 5.5)); plot_df = neo_nodes.head(15).sort_values('n')
    ax.barh(plot_df['label'], plot_df['n'], color=COLORS['orange']); ax.set_title('N?uds Neo4j par label'); ax.set_xlabel('Nombre de n?uds'); ax.grid(axis='x', alpha=0.25); save_fig(fig, '11_neo4j_node_counts')
if not neo_totals.empty:
    save_table(neo_totals, '11_neo4j_totals'); display(neo_totals)

,entity_kind,n,n_with_neo4j_id
0,OFFRE_EMPLOI,15722,15722
1,COMPETENCE,13939,13939
2,METIER,3039,3039
3,CANDIDAT,1105,1105
4,GROUPE_BASE_MEPC,209,209
5,DOMAINE_DETAILLE_NCF,201,201


,label,n
0,GroupeCompétences,14579
1,Compétence,13939
2,OffreEmploi,7861
3,Métier,3039
4,Candidat,1105
5,Employeur,956
6,GroupeISCO,619
7,GroupeBaseMEPC,209
8,DomaineDétailléNCF,201
9,Secteur,190


,metric,n
0,nodes,43008.0
1,relationships,NaN


## 12. Synth?se interpr?table pour le chapitre

Cette cellule produit un r?sum? JSON et un tableau de messages cl?s qui peuvent ?tre repris dans le m?moire sans parler de scripts.

In [13]:
summary = {
    'n_offres': int(len(offres)),
    'n_candidats': int(len(candidats)),
    'n_groupes_base_mepc': int(len(mepc)),
    'n_domaines_detailles_ncf': int(len(ncf)),
    'top_secteur_offres': sector.iloc[0].to_dict() if not sector.empty else None,
    'top_ville_offres': city.iloc[0].to_dict() if not city.empty else None,
    'top_competence_offres': skills.iloc[0].to_dict() if not skills.empty else None,
    'n_pgvector_embeddings': int(pg_counts['n'].sum()) if not pg_counts.empty and 'n' in pg_counts else None,
    'n_neo4j_nodes': int(neo_totals.loc[neo_totals['metric'].eq('nodes'), 'n'].iloc[0]) if not neo_totals.empty and neo_totals['metric'].eq('nodes').any() else None,
    'note_neo4j_relations': 'Le comptage global des relations n?est pas retenu si la base retourne une erreur interne.',
}
with open(OUT / '12_resume_chapitre_implementation.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)
messages = pd.DataFrame([
    {'point': 'Corpus d?offres', 'message': f"Le syst?me s?appuie sur {summary['n_offres']:,} offres normalis?es."},
    {'point': 'Corpus candidats', 'message': f"Le vivier analys? contient {summary['n_candidats']:,} profils candidats normalis?s."},
    {'point': 'R?f?rentiels', 'message': f"Les r?f?rentiels int?gr?s comprennent {summary['n_groupes_base_mepc']} groupes de base MEPC et {summary['n_domaines_detailles_ncf']} domaines d?taill?s NCF."},
    {'point': 'Base vectorielle', 'message': f"La base pgvector contient {summary['n_pgvector_embeddings']:,} embeddings." if summary['n_pgvector_embeddings'] else 'La composition pgvector doit ?tre v?rifi?e.'},
    {'point': 'Graphe', 'message': f"Le graphe Neo4j contient {summary['n_neo4j_nodes']:,} n?uds." if summary['n_neo4j_nodes'] else 'Le nombre de n?uds Neo4j doit ?tre v?rifi?.'},
])
save_table(messages, '12_messages_cles_pour_chapitre')
display(messages)
print(json.dumps(summary, ensure_ascii=False, indent=2))

,point,message
0,Corpus d?offres,"Le syst?me s?appuie sur 7,861 offres normalis?es."
1,Corpus candidats,"Le vivier analys? contient 1,105 profils candi..."
2,R?f?rentiels,Les r?f?rentiels int?gr?s comprennent 209 grou...
3,Base vectorielle,"La base pgvector contient 34,215 embeddings."
4,Graphe,"Le graphe Neo4j contient 43,008 n?uds."


{
  "n_offres": 7861,
  "n_candidats": 1105,
  "n_groupes_base_mepc": 209,
  "n_domaines_detailles_ncf": 201,
  "top_secteur_offres": {
    "secteur": "Distribution",
    "n": 546,
    "part": 0.0695
  },
  "top_ville_offres": {
    "ville": "Yaoundé",
    "n": 4049,
    "part": 0.5151
  },
  "top_competence_offres": {
    "competence": "Distribution",
    "n": 546,
    "part_offres": 0.0695
  },
  "n_pgvector_embeddings": 34215,
  "n_neo4j_nodes": 43008,
  "note_neo4j_relations": "Le comptage global des relations n?est pas retenu si la base retourne une erreur interne."
}
